# cTreeBalls minimal Google Colab example

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/rodriguezmeza/cTreeBalls/blob/main/examples/cTreeBalls_minimal_colab.ipynb)

This notebook is a self-contained Colab version of the repository's Python wrapper tutorial. It installs the published `cTreeBalls` package from PyPI, imports the compiled `cyballs` extension, runs a compact synthetic-catalog calculation, and plots the returned radial bins and 2PCF-style outputs.

No repository checkout or external data download is required.

## 1. Install cTreeBalls

`cTreeBalls` builds a native extension during installation. On Colab this can take a few minutes the first time because the C sources and bundled libraries are compiled locally.

In [ ]:
# Colab normally includes these tools, but installing them explicitly makes
# the notebook easier to run from a fresh runtime.
!sudo apt-get update -qq
!sudo apt-get install -y -qq build-essential python3-dev zlib1g-dev

%pip install --no-cache-dir "cTreeBalls==1.0.1"

## 2. Import the Python wrapper

The PyPI package name is `cTreeBalls`, while the compiled extension is imported as `cyballs`.

In [ ]:
from importlib.metadata import version
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

from cyballs import cballs

print("cTreeBalls package version:", version("cTreeBalls"))
print("Python wrapper class:", cballs)

## 3. Run a compact synthetic example

This follows the small Python-wrapper example in the documentation. The run generates a compact synthetic catalog internally, computes histogram outputs, copies the arrays into NumPy, and releases the C-owned memory with `clean_all()`.

The parameters below keep the runtime short for Colab. Increase `nbody`, `sizeHistN`, or `numberThreads` after the basic workflow runs successfully.

In [ ]:
output_dir = Path("Output_cTreeBalls_colab")

model = cballs()
model.set(
    nbody=4096,
    sizeHistN=12,
    mChebyshev=3,
    rootDir=str(output_dir),
    numberThreads=1,
    verbose=0,
    verbose_log=0,
)

try:
    cpu_time = model.Run()
    radius = np.array(model.getrBins(), copy=True)
    xi = np.array(model.getHistXi2pcf(), copy=True)
    pair_counts = np.array(model.getHistNN(), copy=True)
    native_version = model.getVersion().decode()
finally:
    model.clean_all()

print(f"Native cTreeBalls version: {native_version}")
print(f"CPU time: {cpu_time:.3f} s")
print("Number of radial bins:", len(radius))
print("First five radial bins:", radius[:5])
print("First five xi values:", xi[:5])
print("First five pair counts:", pair_counts[:5])

## 4. Inspect and plot the outputs

`radius`, `xi`, and `pair_counts` are ordinary NumPy arrays after they are copied out of the C wrapper.

In [ ]:
summary = np.column_stack([radius, xi, pair_counts])
np.set_printoptions(precision=6, suppress=False)
print(" columns: radius, xi, pair_counts")
print(summary)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(radius, xi, marker="o")
axes[0].axhline(0, color="0.7", lw=1)
axes[0].set_xscale("log")
axes[0].set_xlabel("radial bin")
axes[0].set_ylabel("xi")
axes[0].set_title("2PCF-style output")
axes[0].grid(alpha=0.3)

axes[1].bar(np.arange(len(pair_counts)), pair_counts)
axes[1].set_xlabel("bin index")
axes[1].set_ylabel("pair count")
axes[1].set_title("Histogram counts")
axes[1].grid(axis="y", alpha=0.3)

plt.tight_layout()

## 5. Check generated files

The run writes logs and parameter records under `Output_cTreeBalls_colab`. These files are useful for reproducing a run with the command-line executable or for checking exactly which parameters were consumed.

In [ ]:
!find Output_cTreeBalls_colab -maxdepth 3 -type f | sort | head -30

## Next steps

- Replace the synthetic run with a catalog workflow by setting `infile`, `infileformat`, and `iCatalogs`.
- Use `options="compute-HistN"` or the 3PCF options used in the repository tests when moving to real data.
- Keep copying arrays into NumPy before calling `clean_all()`.

In [ ]:
# Optional Colab download helper.
# Run this if you want to download the generated output directory.
try:
    from google.colab import files
    !zip -qr cTreeBalls_colab_output.zip Output_cTreeBalls_colab
    files.download("cTreeBalls_colab_output.zip")
except Exception as exc:
    print("Download helper is only available inside Google Colab:", exc)